In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import csv
import time
from selenium.webdriver.chrome.service import Service

In [ ]:
# Define the LinkedIn job search URL
job = 'data'
location = 'belgium'
search_url = f'https://www.linkedin.com/jobs/search/?keywords={job}&location={location}'

In [ ]:
import csv
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException, StaleElementReferenceException

# Define the LinkedIn job search URL
job = 'data'
location = 'belgium'
search_url = f'https://www.linkedin.com/jobs/search/?keywords={job}&location={location}'

# Set up the Chrome WebDriver service
s = Service('/Users/ayushshrestha/Downloads/chromedriver-mac-arm64/chromedriver')

chrome_options = Options()
chrome_options.add_argument("--start-fullscreen")

driver = webdriver.Chrome(service=s, options=chrome_options)

# Navigate to the LinkedIn job search URL
driver.get(search_url)

# Wait for the page to fully load
time.sleep(5)

# Find job listings
job_listings = driver.find_elements(By.XPATH, '//*[@id="main-content"]/section[2]/ul/li')

# Create and open a CSV file to write data
with open('job_listings4.csv', 'w', newline='', encoding='utf-8') as csvfile:
    csv_writer = csv.writer(csvfile)
    csv_writer.writerow(['Title', 'Address', 'Link', 'Job Description'])

    # Loop through job listings to extract information
    for i, listing in enumerate(job_listings):
        try:
            # Click on the listing to reveal more details
            listing.click()
            
            # Wait for job details to load after clicking
            time.sleep(5)

            # Extract job title
            job_title_element = listing.find_element(By.CLASS_NAME, 'sr-only')
            job_title = job_title_element.text

            # Extract the address, getting only the first line
            address_element = listing.find_element(By.CLASS_NAME, 'base-search-card__metadata')
            address_full = address_element.text
            address = address_full.split('\n')[0]

            # Extract the job link
            link_element = listing.find_element(By.CLASS_NAME, 'base-card__full-link')
            job_link = link_element.get_attribute('href')

            # Try to extract the job description
            job_description = "No description found"  # Default message
            try:
                # Check for "show more" button and click it
                try:
                    show_more_button = driver.find_element(By.CLASS_NAME, 'show-more-less-html__button')
                    show_more_button.click()
                    time.sleep(2)  # Wait for the full description to load
                except NoSuchElementException:
                    pass  # If no "show more" button, continue

                # Find the job description with specified class name
                description_element = driver.find_element(By.CLASS_NAME, 'decorated-job-posting__details')
                job_description = description_element.text.replace('\n', ' ')
            except NoSuchElementException:
                print(f"Job description not found for job: {job_title}")

            # Write the extracted data to the CSV file
            csv_writer.writerow([job_title, address, job_link, job_description])

            # Allow time for the page to load after navigating back
            time.sleep(2)

        except (ElementClickInterceptedException, StaleElementReferenceException) as e:
            print(f"Error interacting with job listing {i}: {e}")
            continue

# Close the WebDriver
driver.quit()